# Serveur TripoSR gratuit dans Google Colab

Ce notebook démarre le code officiel TripoSR sur le GPU temporaire fourni par Colab et expose son interface Gradio par un lien public temporaire. Il n'installe ni PyTorch, ni CUDA, ni TripoSR sur l'ordinateur local et ne modifie pas `backend/requirements.txt`.

## Avant de lancer
1. Dans Colab, choisissez **Exécution > Modifier le type d'exécution > T4 GPU**.
2. Exécutez les cellules dans l'ordre. Si l'installation officielle échoue explicitement sur `torchmcubes`, lancez une seule fois la cellule de dépannage dédiée.
3. Exécutez la cellule de compatibilité Colab avant le serveur.
4. Gardez la dernière cellule en cours d'exécution pendant les essais et une éventuelle démonstration.
5. Copiez le lien public temporaire affiché par la dernière cellule et communiquez-le sous la forme `URL : https://...`.

Le serveur n'a pas d'authentification dans cette configuration : toute personne qui obtient le lien peut appeler le modèle. N'envoyez que des photos libres de droits, sans personne ni donnée personnelle. Colab gratuit ne garantit ni un GPU T4, ni la disponibilité, ni une durée fixe. La session peut s'arrêter après inactivité ou lorsque ses limites dynamiques sont atteintes.

**Statut : testé sur Colab le 21 octobre 2026.** Le runtime historique `2025.10` a fourni Python 3.12.12, Tesla T4 et PyTorch CUDA 12.6. Le dépôt officiel a été cloné, les dépendances ont été installées et la cellule de dépannage a compilé `torchmcubes`. La cellule de compatibilité a installé `onnxruntime` CPU et `huggingface-hub==0.34.0`. Le serveur Gradio a démarré et l'application Forma3D a reçu un GLB TripoSR réel. Le lien n'est volontairement pas enregistré dans ce notebook : il est temporaire et public. Les sorties des cellules doivent rester effacées dans le fichier versionné.

Références officielles : README, `requirements.txt`, `gradio_app.py` et `run.py` du dépôt VAST-AI-Research/TripoSR. Le dépôt et ses poids pré-entraînés sont publiés sous licence MIT.


In [ ]:
# Rôle : vérifie le GPU T4 et identifie la version Python du runtime Colab.
# Exécution : première cellule de code, avant le clonage et les installations.
import subprocess
import sys
import torch

subprocess.run(["nvidia-smi"], check=True)
print(f"Version Python de l'image Colab : {sys.version.split()[0]}")
if not torch.cuda.is_available():
    raise RuntimeError("Aucun GPU CUDA n'est actif. Choisissez T4 GPU dans les paramètres du runtime.")
gpu_name = torch.cuda.get_device_name(0)
print(f"GPU détecté : {gpu_name}")
print(f"Version CUDA fournie avec PyTorch : {torch.version.cuda}")
if "T4" not in gpu_name.upper():
    raise RuntimeError("Ce notebook attend un GPU T4. Vérifiez le type de runtime avant de continuer.")
print("Vérification T4 réussie. PyTorch vient du runtime Colab ; aucun PyTorch local n'est installé.")

In [ ]:
# Rôle : récupère le dépôt officiel sans écraser un clonage déjà présent.
# Exécution : après vérification du GPU, depuis le runtime Colab temporaire.
from pathlib import Path
import subprocess

repository = Path("/content/TripoSR")
if not repository.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/VAST-AI-Research/TripoSR.git", str(repository)],
        check=True,
    )
elif not (repository / ".git").exists():
    raise RuntimeError(f"{repository} existe mais ne contient pas un dépôt Git ; rien n'a été remplacé.")
else:
    print(f"Dépôt déjà présent : {repository}")
print(f"Code officiel prêt : {repository}")

In [ ]:
# Rôle : installe dans le runtime Colab les dépendances documentées par TripoSR.
# Exécution : après le clonage ; cette installation ne touche pas l'ordinateur local.
import subprocess
import sys

repository = "/content/TripoSR"
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "setuptools"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", f"{repository}/requirements.txt"],
    check=True,
)
print("Installation des dépendances terminée dans le runtime Colab.")

## Dépannage ciblé : erreur CUDA de torchmcubes

N’exécutez la cellule suivante que si l’installation ou l’application échoue explicitement sur `torchmcubes`. Le README de TripoSR donne une commande de réinstallation simple. Le README actuel de `tatsy/torchmcubes` demande PyTorch déjà installé, un compilateur C++20, CMake 3.24 ou plus et, pour le support GPU, CUDA Toolkit 12 ou plus. Il recommande d’installer `scikit-build-core` et `pybind11`, puis de compiler avec `--no-build-isolation` afin de réutiliser le PyTorch du runtime. Sans CUDA Toolkit, le projet annonce une compilation CPU uniquement. Cette cellule affiche stdout/stderr en cas d’échec et ne modifie que Colab.


In [ ]:
# Rôle : réinstalle torchmcubes avec ses prérequis de compilation officiels et conserve le diagnostic pip.
# Exécution : uniquement après un échec explicitement lié à torchmcubes, dans le runtime Colab.
import subprocess
import sys

def installer_et_afficher_erreur(command):
    print("Commande :", " ".join(command), flush=True)
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, end="" if result.stdout.endswith("\n") else "\n")
    if result.stderr:
        print("--- stderr pip ---")
        print(result.stderr, end="" if result.stderr.endswith("\n") else "\n")
    if result.returncode != 0:
        raise RuntimeError(f"La commande pip a échoué (code {result.returncode}). Consultez stdout et stderr ci-dessus.")

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchmcubes"], check=False)
installer_et_afficher_erreur([sys.executable, "-m", "pip", "install", "scikit-build-core", "pybind11", "cmake"])
installer_et_afficher_erreur([
    sys.executable, "-m", "pip", "install", "--no-build-isolation",
    "git+https://github.com/tatsy/torchmcubes.git",
])
print("Réinstallation terminée. Vérifiez les importations avant de démarrer TripoSR.")

In [ ]:
# Rôle : installe les deux compléments nécessaires dans le runtime Colab observé.
# Exécution : après requirements.txt (et le dépannage torchmcubes si nécessaire), avant Gradio.
# Pourquoi : rembg charge onnxruntime ; le hub fourni par requirements.txt est incompatible avec accelerate préinstallé.
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "onnxruntime", "huggingface-hub==0.34.0"],
    check=True,
)

import onnxruntime
import torch
import torchmcubes
from huggingface_hub import split_torch_state_dict_into_shards
from transformers.models.vit.modeling_vit import ViTModel

print(f"onnxruntime CPU : {onnxruntime.__version__}")
print(f"torchmcubes : {torchmcubes.__file__}")
print(f"huggingface-hub : {split_torch_state_dict_into_shards.__module__}")
print(f"ViTModel : {ViTModel.__name__}")
print(f"GPU prêt : {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'aucun'})")


## Démarrer le serveur et copier le lien

Le code officiel expose réellement les options --share, --port, --username et --password. Le port par défaut est 7860. Cette cellule utilise --share et --port 7860 ; elle ne fournit ni identifiant ni mot de passe, donc le lanceur configure auth=None. Elle affiche le lien public dès que Gradio l'annonce, puis reste active tant que le serveur fonctionne.

N'arrêtez pas cette cellule pendant une génération ou une démonstration. Arrêter la cellule ou perdre le runtime coupe le lien.

In [ ]:
# Rôle : lance gradio_app.py avec un tunnel public et affiche l'URL communiquée par Gradio.
# Exécution : dernière cellule ; la garder active pendant toute utilisation du serveur.
from IPython.display import Markdown, display
from pathlib import Path
import re
import subprocess
import sys

repository = Path("/content/TripoSR")
command = [
    sys.executable,
    "-u",
    "gradio_app.py",
    "--share",
    "--port",
    "7860",
    "--queuesize",
    "1",
]
print("Démarrage de TripoSR ; attendez l'affichage du lien public.", flush=True)
server = subprocess.Popen(
    command,
    cwd=repository,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
public_url = None
interrupted = False
try:
    assert server.stdout is not None
    for line in server.stdout:
        print(line, end="", flush=True)
        if "gradio.live" in line:
            match = re.search(r"https?://[^\s\"'<>]+", line)
            if match:
                public_url = match.group(0).rstrip(".,)")
                display(Markdown(f"Lien public à copier : [{public_url}]({public_url})"))
                print(f"URL : {public_url}", flush=True)
except KeyboardInterrupt:
    interrupted = True
    print("Arrêt demandé : fermeture du serveur TripoSR.", flush=True)
    server.terminate()
    server.wait(timeout=15)
finally:
    if not interrupted and server.poll() is not None and server.returncode != 0:
        raise RuntimeError(f"gradio_app.py s'est arrêté avec le code {server.returncode}.")
if public_url is None:
    print("Gradio n'a pas encore annoncé de lien public. Consultez les sorties de démarrage.", flush=True)